# 2. Filled-ellipse fraction sensitivity

Shrinking the ellipse changes both feature detection and sample support. These plots retain the net vector and non-cancelling exposure separately.

In [ ]:
from pathlib import Path
import sys
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

HERE = Path.cwd().resolve()
ANALYSIS_ROOT = next((p for p in (HERE, *HERE.parents) if (p / "seacofs_tilt_tools.py").exists()), None)
if ANALYSIS_ROOT is None:
    raise FileNotFoundError("Run inside seacofs_eddy_tilt_analysis or one of its subfolders")
for path in (ANALYSIS_ROOT, HERE):
    if str(path) not in sys.path:
        sys.path.insert(0, str(path))
import seacofs_tilt_tools as tilt
import footprint_tools as ft
sns.set_theme(style="whitegrid", context="notebook")
palette = {"AE":"#c44e52", "CE":"#4c72b0"}

data = ft.load_cache()
filled = data[(data.footprint_kind == "Filled") & (data.pv_averaging == "nonlinear")].copy()
score = ft.footprint_scorecard(data)


In [ ]:
metrics = {
    "PV_grad_topo_mag":"Net topographic magnitude",
    "PV_grad_topo_mean_local_mag":"Mean local topographic magnitude",
    "PV_grad_topo_p90_local_mag":"90th-percentile local magnitude",
    "PV_grad_topo_coherence":"Directional coherence",
}
fig, axes = plt.subplots(2,2, figsize=(13,9), constrained_layout=True)
for ax, (column,label) in zip(axes.flat, metrics.items()):
    summary = ft.eddy_equal_summary(filled,column,groups=("Cyc","ellipse_frac"))
    for cyc, part in summary.groupby("Cyc"):
        ax.plot(part.ellipse_frac,part["median"],marker="o",color=palette[cyc],label=cyc)
        ax.fill_between(part.ellipse_frac,part.q25,part.q75,color=palette[cyc],alpha=.16)
    ax.set(xlabel="Ellipse linear fraction",ylabel=label)
    if "mag" in column: ax.set_yscale("log")
axes[0,0].legend(frameon=False); fig.suptitle("Eddy-equal footprint response (IQR shaded)"); plt.show()

In [ ]:
reference = filled[filled.footprint.eq("filled_1")][["Eddy","Day","net_regime"]].rename(columns={"net_regime":"reference_regime"})
change = filled.merge(reference,on=["Eddy","Day"],validate="many_to_one")
change["reclassified"] = change.net_regime.ne(change.reference_regime)
fraction = change.groupby(["Cyc","ellipse_frac"],observed=True).reclassified.mean().reset_index()
fig, ax = plt.subplots(figsize=(9,4.5), constrained_layout=True)
sns.lineplot(data=fraction,x="ellipse_frac",y="reclassified",hue="Cyc",palette=palette,marker="o",ax=ax)
ax.set(xlabel="Ellipse linear fraction",ylabel="Fraction reclassified versus frac=1",title="PV-regime sensitivity to footprint size"); plt.show()